In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/forcewithme/camelyon16-trainr50embedding/mDATA_train.pkl
/kaggle/input/datasets/forcewithme/dsmil-camelyon16-feature/c16-dataset-test/test_026.tif
/kaggle/input/datasets/forcewithme/dsmil-camelyon16-feature/c16-dataset-test/test_104.tif
/kaggle/input/datasets/forcewithme/dsmil-camelyon16-feature/c16-dataset-test/test_068.tif
/kaggle/input/datasets/forcewithme/dsmil-camelyon16-feature/c16-dataset-test/test_016.tif
/kaggle/input/datasets/forcewithme/dsmil-camelyon16-feature/c16-dataset/0-normal.csv
/kaggle/input/datasets/forcewithme/dsmil-camelyon16-feature/c16-dataset/1-tumor.csv
/kaggle/input/datasets/forcewithme/dsmil-camelyon16-feature/c16-dataset/Camelyon16.csv
/kaggle/input/datasets/forcewithme/dsmil-camelyon16-feature/c16-dataset/0-normal/normal_049.csv
/kaggle/input/datasets/forcewithme/dsmil-camelyon16-feature/c16-dataset/0-normal/normal_126.csv
/kaggle/input/datasets/forcewithme/dsmil-camelyon16-feature/c16-dataset/0-normal/test_080.csv
/kaggle/input/data

In [2]:
import os, json, datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, accuracy_score,
                             confusion_matrix, roc_curve)
from sklearn.calibration import calibration_curve
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

BASE = "/kaggle/input/datasets/forcewithme/dsmil-camelyon16-feature/c16-dataset"
CSV  = f"{BASE}/Camelyon16.csv"
OUT  = "/kaggle/working"
os.makedirs(f"{OUT}/checkpoints", exist_ok=True)
os.makedirs(f"{OUT}/plots",       exist_ok=True)
os.makedirs(f"{OUT}/metrics",     exist_ok=True)

MAX_PATCHES = 512
SEEDS       = [42, 123, 456]   # 3 seeds for stability
EPOCHS      = 20
LR          = 1e-4
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device : {DEVICE}")
print(f"Seeds  : {SEEDS}")
print(f"Epochs : {EPOCHS}")

Device : cuda
Seeds  : [42, 123, 456]
Epochs : 20


In [3]:
class Camelyon16BagDataset(Dataset):
    def __init__(self, records, base_dir, max_patches=512, seed=42):
        self.records     = records
        self.base_dir    = base_dir
        self.max_patches = max_patches
        self.rng         = np.random.RandomState(seed)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rel_path, label = self.records[idx]
        rel_path  = rel_path.replace("datasets/Camelyon16/", "")
        full_path = os.path.join(self.base_dir, rel_path)
        feats = torch.tensor(
            np.loadtxt(full_path, delimiter=',', skiprows=1),
            dtype=torch.float32)
        N = feats.shape[0]
        if N > self.max_patches:
            chosen = self.rng.choice(N, self.max_patches, replace=False)
            feats  = feats[chosen]
        return {"features": feats,
                "label":    torch.tensor(label, dtype=torch.long),
                "path":     rel_path}

def bag_collate(batch):
    return {"features": [b["features"] for b in batch],
            "labels":   torch.stack([b["label"] for b in batch]),
            "paths":    [b["path"] for b in batch]}

def make_loaders(seed):
    df_meta         = pd.read_csv(CSV)
    df_meta.columns = ["path", "label"]
    df_meta         = df_meta[df_meta["label"].isin([0,1])].reset_index(drop=True)
    records         = list(zip(df_meta["path"], df_meta["label"]))

    train_val, test_rec = train_test_split(
        records, test_size=0.25, random_state=seed,
        stratify=[r[1] for r in records])
    train_rec, val_rec = train_test_split(
        train_val, test_size=0.15, random_state=seed,
        stratify=[r[1] for r in train_val])

    train_ds = Camelyon16BagDataset(train_rec, BASE, MAX_PATCHES, seed)
    val_ds   = Camelyon16BagDataset(val_rec,   BASE, MAX_PATCHES, seed)
    test_ds  = Camelyon16BagDataset(test_rec,  BASE, MAX_PATCHES, seed+1)

    return (DataLoader(train_ds, batch_size=1, shuffle=True,
                       collate_fn=bag_collate, num_workers=2),
            DataLoader(val_ds,   batch_size=1, shuffle=False,
                       collate_fn=bag_collate, num_workers=2),
            DataLoader(test_ds,  batch_size=1, shuffle=False,
                       collate_fn=bag_collate, num_workers=2),
            test_ds)

print("Dataset loader ready")

Dataset loader ready


In [4]:
COLORS = {
    'ABMIL':                     '#2196F3',
    'CLAM-SB':                   '#4CAF50',
    'TransMIL':                  '#FF5722',
    'VAEABMIL':                  '#FF9800',
    'eval_nogate (No Gating)':   '#9C27B0',
    'URAT-MIL (Proposed)':       '#F44336',
}

def fname(n):
    return n.replace(' ','_').replace('(','').replace(')','').replace('/','_')

def plot_training(losses, aucs, name, seed):
    fig,(ax1,ax2) = plt.subplots(1,2,figsize=(10,4))
    ep = range(1, len(losses)+1)
    ax1.plot(ep, losses, color=COLORS.get(name,'gray'), linewidth=2)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Training Loss')
    ax1.set_title(f'Training Loss — {name} (seed={seed})'); ax1.grid(alpha=0.3)
    ax2.plot(ep, aucs, color='#F44336', linewidth=2)
    best_ep = int(np.argmax(aucs))+1
    ax2.axvline(best_ep, color='gray', linestyle='--', alpha=0.7,
                label=f'Best Ep={best_ep}: {max(aucs):.4f}')
    ax2.legend(fontsize=8)
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val AUC')
    ax2.set_title(f'Validation AUC — {name} (seed={seed})'); ax2.grid(alpha=0.3)
    plt.tight_layout()
    path = f"{OUT}/plots/training_{fname(name)}_seed{seed}.png"
    plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()

def plot_confusion(tn,fp,fn,tp, name, seed):
    cm = np.array([[tn,fp],[fn,tp]])
    fig,ax = plt.subplots(figsize=(5,4))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Predicted\nNormal','Predicted\nTumor'], fontsize=11)
    ax.set_yticklabels(['Actual\nNormal','Actual\nTumor'], fontsize=11)
    for i in range(2):
        for j in range(2):
            ax.text(j,i,str(cm[i,j]),ha='center',va='center',
                    fontsize=16,fontweight='bold',
                    color='white' if cm[i,j]>cm.max()/2 else 'black')
    ax.set_title(f'Confusion — {name} (seed={seed})', fontsize=11)
    plt.colorbar(im,ax=ax); plt.tight_layout()
    path = f"{OUT}/plots/confusion_{fname(name)}_seed{seed}.png"
    plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()

def plot_all_roc(roc_dict, seed):
    fig,ax = plt.subplots(figsize=(7,6))
    for name,(fpr,tpr,auc_val) in roc_dict.items():
        ax.plot(fpr,tpr,label=f"{name} (AUC={auc_val:.4f})",
                color=COLORS.get(name,'gray'),linewidth=2)
    ax.plot([0,1],[0,1],'k--',linewidth=1,label='Random')
    ax.set_xlabel('False Positive Rate',fontsize=12)
    ax.set_ylabel('True Positive Rate',fontsize=12)
    ax.set_title(f'ROC Curves — Seed {seed}',fontsize=13)
    ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout()
    path = f"{OUT}/plots/roc_all_seed{seed}.png"
    plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")

def plot_comparison_bar(mean_results, std_results):
    names = [r['Model'] for r in mean_results]
    aucs  = [r['AUC']   for r in mean_results]
    eces  = [r['ECE']   for r in mean_results]
    auc_e = [std_results[r['Model']]['AUC'] for r in mean_results]
    ece_e = [std_results[r['Model']]['ECE'] for r in mean_results]
    x = np.arange(len(names))

    fig,(ax1,ax2) = plt.subplots(1,2,figsize=(14,5))
    bars = ax1.bar(x, aucs, yerr=auc_e, capsize=4,
                   color=[COLORS.get(n,'gray') for n in names], alpha=0.85)
    ax1.set_ylabel('AUC ',fontsize=11)
    ax1.set_title('AUC Comparison (3 seeds)',fontsize=12)
    ax1.set_xticks(x); ax1.set_xticklabels(names,rotation=25,ha='right',fontsize=8)
    ax1.set_ylim(0.85,1.0); ax1.grid(axis='y',alpha=0.3)
    for bar,v,e in zip(bars,aucs,auc_e):
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+e+0.001,
                 f'{v:.4f}',ha='center',va='bottom',fontsize=7)

    bars2 = ax2.bar(x, eces, yerr=ece_e, capsize=4,
                    color=[COLORS.get(n,'gray') for n in names], alpha=0.85)
    ax2.set_ylabel('ECE ',fontsize=11)
    ax2.set_title('ECE Comparison (3 seeds)',fontsize=12)
    ax2.set_xticks(x); ax2.set_xticklabels(names,rotation=25,ha='right',fontsize=8)
    ax2.grid(axis='y',alpha=0.3)
    for bar,v,e in zip(bars2,eces,ece_e):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+e+0.002,
                 f'{v:.4f}',ha='center',va='bottom',fontsize=7)

    plt.suptitle('Multi-Seed Model Comparison — CAMELYON16',fontsize=13)
    plt.tight_layout()
    path = f"{OUT}/plots/comparison_multiseed.png"
    plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")

def plot_attention_comparison(feats_normal, attn_standard, attn_gated,
                               feats_tumor, attn_std_t, attn_gated_t):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    for row, (feats, a_std, a_gate, label) in enumerate([
        (feats_normal, attn_standard, attn_gated, 'Normal Slide'),
        (feats_tumor,  attn_std_t,    attn_gated_t, 'Tumor Slide'),
    ]):
        n = len(a_std)
        patch_idx = np.arange(n)
        axes[row,0].bar(patch_idx, a_std.cpu().numpy(), color='#2196F3', alpha=0.7, width=1.0)
        axes[row,0].set_title(f'Standard Attention — {label}', fontsize=11)
        axes[row,0].set_xlabel('Patch Index'); axes[row,0].set_ylabel('Attention Weight')
        axes[row,0].grid(alpha=0.3)

        axes[row,1].bar(patch_idx, a_gate.cpu().numpy(), color='#F44336', alpha=0.7, width=1.0)
        axes[row,1].set_title(f'Uncertainty-Gated Attention — {label}', fontsize=11)
        axes[row,1].set_xlabel('Patch Index'); axes[row,1].set_ylabel('Gated Attention Weight')
        axes[row,1].grid(alpha=0.3)

    plt.suptitle('Attention Weight Distribution: Standard vs Uncertainty-Gated', fontsize=13)
    plt.tight_layout()
    path = f"{OUT}/plots/attention_comparison.png"
    plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")

print("Plot utilities ready")

Plot utilities ready


In [5]:
def run_model(model, train_loader, val_loader, test_loader,
              epochs=EPOCHS, lr=LR, name='Model',
              is_vae=False, is_urat=False,
              lam_kl=0.005, lam_recon=0.005, lam_ood=0.005,
              save_path=None, seed=42):

    model = model.to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs, eta_min=1e-6)
    best_auc, best_state = 0.0, None
    losses, val_aucs = [], []

    for ep in range(1, epochs+1):
        model.train()
        total = 0
        beta  = min(1.0, ep/15) * lam_kl if (is_vae or is_urat) else 0

        for batch in train_loader:
            feats  = batch["features"][0].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            if is_urat:
                logits,kl,recon,ood_s,_,_ = model(feats)
                loss = (F.cross_entropy(logits,labels,label_smoothing=0.05)
                        + beta*kl + lam_recon*recon + lam_ood*ood_s)
            elif is_vae:
                logits,kl,recon = model(feats)
                loss = (F.cross_entropy(logits,labels,label_smoothing=0.05)
                        + beta*kl + lam_recon*recon)
            else:
                out = model(feats)
                if isinstance(out,tuple): out = out[0]
                loss = F.cross_entropy(out,labels,label_smoothing=0.05)

            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step()
            total += loss.item()
        sched.step()
        losses.append(total/len(train_loader))

        model.eval()
        vp,vt = [],[]
        with torch.no_grad():
            for batch in val_loader:
                feats = batch["features"][0].to(DEVICE)
                if is_urat:
                    out,*_ = model(feats)
                elif is_vae:
                    out,_,_ = model(feats)
                else:
                    out = model(feats)
                    if isinstance(out,tuple): out = out[0]
                vp.append(torch.softmax(out,dim=1)[0,1].item())
                vt.append(batch["labels"].item())
        vauc = roc_auc_score(vt,vp)
        val_aucs.append(vauc)
        if vauc > best_auc:
            best_auc   = vauc
            best_state = {k:v.clone() for k,v in model.state_dict().items()}
        if ep % 10 == 0:
            print(f"  [{name}|s{seed}] Ep {ep}/{epochs} | "
                  f"Loss:{losses[-1]:.4f} | ValAUC:{vauc:.4f} | Best:{best_auc:.4f}")

    plot_training(losses, val_aucs, name, seed)
    model.load_state_dict(best_state)
    model.eval()

    tp_list,tt,ualeas,uepiss = [],[],[],[]
    with torch.no_grad():
        for batch in test_loader:
            feats = batch["features"][0].to(DEVICE)
            if is_urat:
                out,_,_,_,ua,ue = model(feats)
                ualeas.append(ua.item()); uepiss.append(ue.item())
            elif is_vae:
                out,_,_ = model(feats)
            else:
                out = model(feats)
                if isinstance(out,tuple): out=out[0]
            tp_list.append(torch.softmax(out,dim=1)[0,1].item())
            tt.append(batch["labels"].item())

    preds       = [1 if p>0.5 else 0 for p in tp_list]
    acc         = accuracy_score(tt,preds)
    auc         = roc_auc_score(tt,tp_list)
    tn,fp,fn,tp = confusion_matrix(tt,preds).ravel()
    prec        = tp/(tp+fp) if (tp+fp)>0 else 0
    rec         = tp/(tp+fn) if (tp+fn)>0 else 0
    fpr_c,tpr_c,_ = roc_curve(tt,tp_list)
    fp2,mc      = calibration_curve(tt,tp_list,n_bins=10)
    ece         = float(np.mean(np.abs(fp2-mc)))

    plot_confusion(int(tn),int(fp),int(fn),int(tp),name,seed)

    print(f"\n  {'='*45}")
    print(f"  {name} | seed={seed}")
    print(f"  Accuracy:{acc*100:.2f}% AUC:{auc:.4f} ECE:{ece:.4f}")
    print(f"  Precision:{prec:.4f} Recall:{rec:.4f}")
    if is_urat:
        print(f"  U_alea:{np.mean(ualeas):.4f} U_epis:{np.mean(uepiss):.4f}")
    print(f"  TP={tp} TN={tn} FP={fp} FN={fn}")
    print(f"  {'='*45}\n")

    if save_path: torch.save(best_state, save_path)

    return {"Model":name,"Accuracy":round(acc*100,2),
            "AUC":round(auc,4),"ECE":round(ece,4),
            "Precision":round(prec,4),"Recall":round(rec,4),
            "TP":int(tp),"TN":int(tn),"FP":int(fp),"FN":int(fn),
            "U_alea":round(float(np.mean(ualeas)),4) if ualeas else "N/A",
            "U_epis":round(float(np.mean(uepiss)),4) if uepiss else "N/A",
            "_fpr":fpr_c.tolist(),"_tpr":tpr_c.tolist(),
            "_fp2":fp2.tolist(),"_mc":mc.tolist(),
            "_losses":losses,"_aucs":val_aucs,
            "seed":seed}

print("run_model ready")

run_model ready


In [6]:
# ABMIL 
class ABMIL(nn.Module):
    def __init__(self, feat_dim=512, hidden=256, att_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(feat_dim,hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(0.25))
        self.att_V = nn.Linear(hidden,att_dim)
        self.att_U = nn.Linear(hidden,att_dim)
        self.att_w = nn.Linear(att_dim,1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden,64), nn.GELU(),
            nn.Dropout(0.25), nn.Linear(64,2))

    def forward(self, feats):
        h = self.encoder(feats)
        a = torch.tanh(self.att_V(h))*torch.sigmoid(self.att_U(h))
        a = torch.softmax(self.att_w(a),dim=0)
        return self.classifier((a*h).sum(dim=0,keepdim=True))

#CLAM-SB 
class CLAM_SB(nn.Module):
    def __init__(self, feat_dim=512, hidden=256, att_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(feat_dim,hidden), nn.ReLU(), nn.Dropout(0.25))
        self.att_V = nn.Linear(hidden,att_dim)
        self.att_U = nn.Linear(hidden,att_dim)
        self.att_w = nn.Linear(att_dim,1)
        self.classifier = nn.Linear(hidden,2)

    def forward(self, feats):
        h = self.encoder(feats)
        a = torch.tanh(self.att_V(h))*torch.sigmoid(self.att_U(h))
        a = torch.softmax(self.att_w(a),dim=0)
        return self.classifier((a*h).sum(dim=0,keepdim=True))

# TransMIL 
class TransMIL(nn.Module):
    def __init__(self, feat_dim=512, hidden=256, nhead=4,
                 num_layers=2, dropout=0.25):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(feat_dim, hidden),
            nn.LayerNorm(hidden))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, hidden))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=nhead,
            dim_feedforward=hidden*2,
            dropout=dropout, batch_first=True,
            norm_first=True)
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden,64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64,2))

    def forward(self, feats):
        # feats: (N, feat_dim)
        x   = self.proj(feats).unsqueeze(0)          # (1, N, hidden)
        cls = self.cls_token.expand(1, -1, -1)        # (1, 1, hidden)
        x   = torch.cat([cls, x], dim=1)              # (1, N+1, hidden)
        out = self.transformer(x)                     # (1, N+1, hidden)
        cls_out = out[:, 0, :]                         # (1, hidden) — CLS token
        return self.classifier(cls_out)

# VAEABMIL 
class VAEABMIL(nn.Module):
    def __init__(self, feat_dim=512, hidden=256, latent=128, att_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(feat_dim,hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(0.25))
        self.mu_head     = nn.Linear(hidden,latent)
        self.logvar_head = nn.Linear(hidden,latent)
        self.logvar_head.bias.data.fill_(-2.0)
        self.decoder = nn.Sequential(
            nn.Linear(latent,hidden), nn.GELU(), nn.Linear(hidden,feat_dim))
        self.att_V = nn.Linear(hidden,att_dim)
        self.att_U = nn.Linear(hidden,att_dim)
        self.att_w = nn.Linear(att_dim,1)
        self.classifier = nn.Sequential(
            nn.Linear(latent,64), nn.GELU(),
            nn.Dropout(0.25), nn.Linear(64,2))

    def reparameterize(self, mu, logvar):
        if self.training:
            return mu + torch.exp(0.5*logvar)*torch.randn_like(mu)
        return mu

    def forward(self, feats):
        h      = self.encoder(feats)
        mu     = self.mu_head(h)
        logvar = self.logvar_head(h).clamp(-8,4)
        z      = self.reparameterize(mu,logvar)
        a = torch.tanh(self.att_V(h))*torch.sigmoid(self.att_U(h))
        a = torch.softmax(self.att_w(a),dim=0)
        z_bag  = (a*z).sum(dim=0,keepdim=True)
        logits = self.classifier(z_bag)
        kl     = -0.5*torch.mean(1+logvar-mu.pow(2)-logvar.exp())
        recon  = F.mse_loss(self.decoder(z),feats)
        return logits,kl,recon

# URAT-MIL with LEARNABLE GATING 
class URATMIL(nn.Module):
    def __init__(self, feat_dim=512, hidden=256, latent=128,
                 att_dim=128, mc_samples=20, gate=True):
        super().__init__()
        self.mc_samples = mc_samples
        self.gate       = gate

        self.encoder = nn.Sequential(
            nn.Linear(feat_dim,hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(0.25))
        self.mu_head     = nn.Linear(hidden,latent)
        self.logvar_head = nn.Linear(hidden,latent)
        self.logvar_head.bias.data.fill_(-2.0)
        self.decoder = nn.Sequential(
            nn.Linear(latent,hidden), nn.GELU(), nn.Linear(hidden,feat_dim))
        self.prototypes = nn.Parameter(torch.randn(2,latent))
        self.att_V      = nn.Linear(hidden,att_dim)
        self.att_U      = nn.Linear(hidden,att_dim)
        self.att_w      = nn.Linear(att_dim,1)
        self.temperature= nn.Parameter(torch.ones(1))

        # Maps scalar uncertainty U_total -> gating weight in (0,1)
        # Initialized so sigmoid(W*U+b) ≈ 1 at start (near-identity)
        # so model starts close to standard attention and learns to gate
        self.gate_fc = nn.Linear(1, 1, bias=True)
        nn.init.constant_(self.gate_fc.weight, -2.0)  # start: suppress uncertain patches
        nn.init.constant_(self.gate_fc.bias,    2.0)  # start: gate ≈ sigmoid(2) ≈ 0.88

        self.classifier = nn.Sequential(
            nn.LayerNorm(latent), nn.Linear(latent,128),
            nn.GELU(), nn.Dropout(0.25), nn.Linear(128,2))

    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5*logvar)*torch.randn_like(mu)

    def get_attention(self, feats):
        h      = self.encoder(feats)
        mu     = self.mu_head(h)
        logvar = self.logvar_head(h).clamp(-8,4)

        samples = torch.stack([
            self.reparameterize(mu,logvar)
            for _ in range(self.mc_samples)],dim=0)
        z      = samples.mean(dim=0)
        u_alea = torch.exp(logvar).mean(dim=1,keepdim=True)
        u_epis = samples.var(dim=0).mean(dim=1,keepdim=True)
        u_total= u_alea + u_epis

        # Standard attention
        a = torch.tanh(self.att_V(h))*torch.sigmoid(self.att_U(h))
        a = torch.softmax(self.att_w(a),dim=0)  # (N,1)

        if self.gate:
            # LEARNABLE GATING
            gate_w = torch.sigmoid(self.gate_fc(u_total))  # (N,1) in (0,1)
            a_gated = a * gate_w
            a_gated = a_gated / (a_gated.sum() + 1e-8)
        else:
            a_gated = a

        return a.squeeze(1), a_gated.squeeze(1), u_total.squeeze(1)

    def forward(self, feats):
        h      = self.encoder(feats)
        mu     = self.mu_head(h)
        logvar = self.logvar_head(h).clamp(-8,4)

        if self.training:
            z      = self.reparameterize(mu,logvar)
            u_alea = torch.exp(logvar).mean(dim=1,keepdim=True)
            u_epis = torch.zeros_like(u_alea)
        else:
            samples = torch.stack([
                self.reparameterize(mu,logvar)
                for _ in range(self.mc_samples)],dim=0)
            z      = samples.mean(dim=0)
            u_alea = torch.exp(logvar).mean(dim=1,keepdim=True)
            u_epis = samples.var(dim=0).mean(dim=1,keepdim=True)

        u_total = u_alea + u_epis
        a = torch.tanh(self.att_V(h))*torch.sigmoid(self.att_U(h))
        a = torch.softmax(self.att_w(a),dim=0)

        if self.gate:
            # LEARNABLE GATING — the upgrade
            gate_w = torch.sigmoid(self.gate_fc(u_total))
            a      = a * gate_w
            a      = a / (a.sum() + 1e-8)

        z_bag  = (a*z).sum(dim=0,keepdim=True)
        logits = self.classifier(z_bag)/self.temperature.clamp(min=0.1)
        kl     = -0.5*torch.mean(1+logvar-mu.pow(2)-logvar.exp())
        recon  = F.mse_loss(self.decoder(z),feats)
        dists  = torch.cdist(z,self.prototypes)
        ood_s  = dists.min(dim=1).values.mean()
        return logits,kl,recon,ood_s,u_alea.mean(),u_epis.mean()

print("All models defined")

All models defined


In [7]:
# Storage for all results across seeds
all_seed_results = {
    'ABMIL':                   [],
    'CLAM-SB':                 [],
    'TransMIL':                [],
    'VAEABMIL':                [],
    'eval_nogate (No Gating)': [],
    'URAT-MIL (Proposed)':     [],
}

for seed in SEEDS:
    print(f"# SEED {seed}")
    

    train_loader, val_loader, test_loader, test_ds = make_loaders(seed)

    # 1. ABMIL
    r = run_model(ABMIL(), train_loader, val_loader, test_loader,
                  name='ABMIL', seed=seed,
                  save_path=f'{OUT}/checkpoints/abmil_seed{seed}.pt')
    all_seed_results['ABMIL'].append(r)

    # 2. CLAM-SB
    r = run_model(CLAM_SB(), train_loader, val_loader, test_loader,
                  name='CLAM-SB', seed=seed,
                  save_path=f'{OUT}/checkpoints/clam_seed{seed}.pt')
    all_seed_results['CLAM-SB'].append(r)

    # 3. TransMIL (NEW)
    r = run_model(TransMIL(), train_loader, val_loader, test_loader,
                  name='TransMIL', seed=seed,
                  save_path=f'{OUT}/checkpoints/transmil_seed{seed}.pt')
    all_seed_results['TransMIL'].append(r)

    # 4. VAEABMIL
    r = run_model(VAEABMIL(), train_loader, val_loader, test_loader,
                  name='VAEABMIL', is_vae=True, seed=seed,
                  save_path=f'{OUT}/checkpoints/vaeabmil_seed{seed}.pt')
    all_seed_results['VAEABMIL'].append(r)

    # 5. eval_nogate
    r = run_model(URATMIL(gate=False), train_loader, val_loader, test_loader,
                  name='eval_nogate (No Gating)', is_urat=True, seed=seed,
                  save_path=f'{OUT}/checkpoints/nogate_seed{seed}.pt')
    all_seed_results['eval_nogate (No Gating)'].append(r)

    # 6. URAT-MIL with LEARNABLE GATING
    urat_model = URATMIL(gate=True)
    r = run_model(urat_model, train_loader, val_loader, test_loader,
                  name='URAT-MIL (Proposed)', is_urat=True, seed=seed,
                  save_path=f'{OUT}/checkpoints/urat_seed{seed}.pt')
    all_seed_results['URAT-MIL (Proposed)'].append(r)

    # ROC curve for this seed
    roc_dict = {}
    for name, results in all_seed_results.items():
        if len(results) >= SEEDS.index(seed)+1:
            res = results[-1]
            roc_dict[name] = (res['_fpr'], res['_tpr'], res['AUC'])
    plot_all_roc(roc_dict, seed)

    print(f"\n Seed {seed} complete")

# SEED 42
  [ABMIL|s42] Ep 10/20 | Loss:0.4498 | ValAUC:0.8128 | Best:0.8560
  [ABMIL|s42] Ep 20/20 | Loss:0.4532 | ValAUC:0.8395 | Best:0.8560

  ABMIL | seed=42
  Accuracy:88.00% AUC:0.9113 ECE:0.4409
  Precision:0.9375 Recall:0.7500
  TP=30 TN=58 FP=2 FN=10

  [CLAM-SB|s42] Ep 10/20 | Loss:0.4423 | ValAUC:0.8663 | Best:0.8765
  [CLAM-SB|s42] Ep 20/20 | Loss:0.5416 | ValAUC:0.8745 | Best:0.8765

  CLAM-SB | seed=42
  Accuracy:87.00% AUC:0.9150 ECE:0.4040
  Precision:0.9355 Recall:0.7250
  TP=29 TN=58 FP=2 FN=11



/tmp/ipykernel_24/1671224873.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


  [TransMIL|s42] Ep 10/20 | Loss:0.5415 | ValAUC:0.8374 | Best:0.8374
  [TransMIL|s42] Ep 20/20 | Loss:0.5207 | ValAUC:0.8313 | Best:0.8560

  TransMIL | seed=42
  Accuracy:87.00% AUC:0.9292 ECE:0.0879
  Precision:0.9355 Recall:0.7250
  TP=29 TN=58 FP=2 FN=11

  [VAEABMIL|s42] Ep 10/20 | Loss:0.5236 | ValAUC:0.8498 | Best:0.8498
  [VAEABMIL|s42] Ep 20/20 | Loss:0.5349 | ValAUC:0.8416 | Best:0.8498

  VAEABMIL | seed=42
  Accuracy:86.00% AUC:0.9196 ECE:0.5130
  Precision:0.8421 Recall:0.8000
  TP=32 TN=54 FP=6 FN=8

  [eval_nogate (No Gating)|s42] Ep 10/20 | Loss:0.4658 | ValAUC:0.8210 | Best:0.8663
  [eval_nogate (No Gating)|s42] Ep 20/20 | Loss:0.5694 | ValAUC:0.8230 | Best:0.8663

  eval_nogate (No Gating) | seed=42
  Accuracy:89.00% AUC:0.9188 ECE:0.2064
  Precision:1.0000 Recall:0.7250
  U_alea:0.0135 U_epis:0.0135
  TP=29 TN=60 FP=0 FN=11

  [URAT-MIL (Proposed)|s42] Ep 10/20 | Loss:0.5552 | ValAUC:0.8416 | Best:0.8580
  [URAT-MIL (Proposed)|s42] Ep 20/20 | Loss:0.5100 | ValAUC:0.

/tmp/ipykernel_24/1671224873.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


  [TransMIL|s123] Ep 10/20 | Loss:0.6300 | ValAUC:0.8601 | Best:0.8724
  [TransMIL|s123] Ep 20/20 | Loss:0.5030 | ValAUC:0.8786 | Best:0.8786

  TransMIL | seed=123
  Accuracy:90.00% AUC:0.9242 ECE:0.0857
  Precision:0.9688 Recall:0.7750
  TP=31 TN=59 FP=1 FN=9

  [VAEABMIL|s123] Ep 10/20 | Loss:0.5351 | ValAUC:0.8539 | Best:0.8724
  [VAEABMIL|s123] Ep 20/20 | Loss:0.4971 | ValAUC:0.8601 | Best:0.8724

  VAEABMIL | seed=123
  Accuracy:88.00% AUC:0.9267 ECE:0.3616
  Precision:0.9118 Recall:0.7750
  TP=31 TN=57 FP=3 FN=9

  [eval_nogate (No Gating)|s123] Ep 10/20 | Loss:0.5690 | ValAUC:0.8539 | Best:0.8621
  [eval_nogate (No Gating)|s123] Ep 20/20 | Loss:0.6395 | ValAUC:0.8663 | Best:0.8745

  eval_nogate (No Gating) | seed=123
  Accuracy:90.00% AUC:0.9217 ECE:0.1594
  Precision:0.9688 Recall:0.7750
  U_alea:0.0208 U_epis:0.0208
  TP=31 TN=59 FP=1 FN=9

  [URAT-MIL (Proposed)|s123] Ep 10/20 | Loss:0.5373 | ValAUC:0.8519 | Best:0.8621
  [URAT-MIL (Proposed)|s123] Ep 20/20 | Loss:0.5067 | 

/tmp/ipykernel_24/1671224873.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


  [TransMIL|s456] Ep 10/20 | Loss:0.6393 | ValAUC:0.9239 | Best:0.9239
  [TransMIL|s456] Ep 20/20 | Loss:0.5364 | ValAUC:0.9136 | Best:0.9239

  TransMIL | seed=456
  Accuracy:79.00% AUC:0.8950 ECE:0.3937
  Precision:0.6863 Recall:0.8750
  TP=35 TN=44 FP=16 FN=5

  [VAEABMIL|s456] Ep 10/20 | Loss:0.5700 | ValAUC:0.8971 | Best:0.9198
  [VAEABMIL|s456] Ep 20/20 | Loss:0.5491 | ValAUC:0.9095 | Best:0.9198

  VAEABMIL | seed=456
  Accuracy:91.00% AUC:0.9179 ECE:0.2127
  Precision:0.9429 Recall:0.8250
  TP=33 TN=58 FP=2 FN=7

  [eval_nogate (No Gating)|s456] Ep 10/20 | Loss:0.6344 | ValAUC:0.9177 | Best:0.9177
  [eval_nogate (No Gating)|s456] Ep 20/20 | Loss:0.5656 | ValAUC:0.9156 | Best:0.9177

  eval_nogate (No Gating) | seed=456
  Accuracy:89.00% AUC:0.9208 ECE:0.3278
  Precision:0.8718 Recall:0.8500
  U_alea:0.0173 U_epis:0.0173
  TP=34 TN=55 FP=5 FN=6

  [URAT-MIL (Proposed)|s456] Ep 10/20 | Loss:0.5796 | ValAUC:0.9156 | Best:0.9218
  [URAT-MIL (Proposed)|s456] Ep 20/20 | Loss:0.5595 |

In [8]:
print("Generating attention comparison plots...")

urat_viz = URATMIL(gate=True).to(DEVICE)
urat_viz.load_state_dict(
    torch.load(f'{OUT}/checkpoints/urat_seed42.pt', map_location=DEVICE))
urat_viz.eval()

# Load dataset for seed 42
_, _, test_loader_viz, test_ds_viz = make_loaders(42)

# Find one normal and one tumor slide
normal_feats = tumor_feats = None
normal_path  = tumor_path  = ""

for batch in test_loader_viz:
    feats = batch["features"][0]
    label = batch["labels"][0].item()
    path  = batch["paths"][0]
    if label == 0 and normal_feats is None:
        normal_feats = feats; normal_path = path
    if label == 1 and tumor_feats is None:
        tumor_feats = feats; tumor_path = path
    if normal_feats is not None and tumor_feats is not None:
        break

# Get attention weights for both slides
with torch.no_grad():
    a_std_n, a_gate_n, u_n = urat_viz.get_attention(normal_feats.to(DEVICE))
    a_std_t, a_gate_t, u_t = urat_viz.get_attention(tumor_feats.to(DEVICE))

print(f"Normal slide: {normal_path}")
print(f"  Standard attention max: {a_std_n.max().item():.4f}")
print(f"  Gated attention max:    {a_gate_n.max().item():.4f}")
print(f"  Mean uncertainty:       {u_n.mean().item():.4f}")

print(f"\nTumor slide: {tumor_path}")
print(f"  Standard attention max: {a_std_t.max().item():.4f}")
print(f"  Gated attention max:    {a_gate_t.max().item():.4f}")
print(f"  Mean uncertainty:       {u_t.mean().item():.4f}")

plot_attention_comparison(
    normal_feats, a_std_n, a_gate_n,
    tumor_feats,  a_std_t, a_gate_t)

print("Attention plots saved")

Generating attention comparison plots...
Normal slide: 0-normal/normal_023.csv
  Standard attention max: 0.0027
  Gated attention max:    0.0027
  Mean uncertainty:       0.0428

Tumor slide: 1-tumor/test_029.csv
  Standard attention max: 0.1431
  Gated attention max:    0.1412
  Mean uncertainty:       0.0433
  Saved: /kaggle/working/plots/attention_comparison.png
Attention plots saved


In [9]:
metrics = ['AUC','ECE','Accuracy','Precision','Recall']

mean_results = []
std_dict     = {}

for name, results in all_seed_results.items():
    row = {"Model": name}
    std_dict[name] = {}
    for m in metrics:
        vals = [r[m] for r in results if isinstance(r[m], (int,float))]
        if vals:
            row[m]         = round(float(np.mean(vals)), 4)
            std_dict[name][m] = round(float(np.std(vals)),  4)
        else:
            row[m]         = "N/A"
            std_dict[name][m] = 0.0
    mean_results.append(row)

# Print table

print("MULTI-SEED RESULTS ")

header = f"{'Model':<32} {'AUC':>12} {'ECE':>12} {'Acc':>8} {'Prec':>8} {'Rec':>8}"
print(header)

for r in mean_results:
    name = r['Model']
    line = f"{name:<32}"
    for m in metrics[:3]:
        v = r[m]; s = std_dict[name][m]
        line += f"  {v:.4f}±{s:.4f}" if isinstance(v,float) else f"  {'N/A':>11}"
    for m in metrics[3:]:
        v = r[m]
        line += f"  {v:.4f}" if isinstance(v,float) else "  N/A"
    print(line)
print("="*80)

# Plot comparison
plot_comparison_bar(mean_results, std_dict)

# Save all outputs
df_mean = pd.DataFrame(mean_results)
df_mean.to_csv(f"{OUT}/metrics/results_mean.csv", index=False)

df_all = pd.DataFrame([
    {**r, 'model':name}
    for name, results in all_seed_results.items()
    for r in results
])
df_all.to_csv(f"{OUT}/metrics/results_all_seeds.csv", index=False)

with open(f"{OUT}/metrics/all_results.json", 'w') as f:
    json.dump(all_seed_results, f, indent=2, default=str)

meta = {
    "timestamp":   datetime.datetime.now().isoformat(),
    "seeds":       SEEDS,
    "epochs":      EPOCHS,
    "dataset":     "CAMELYON16",
    "gating":      "LEARNABLE sigmoid(W*U+b)",
    "baselines":   ["ABMIL","CLAM-SB","TransMIL","VAEABMIL"],
    "ablations":   ["eval_nogate"],
}
with open(f"{OUT}/metrics/metadata.json",'w') as f:
    json.dump(meta,f,indent=2)

# List all outputs
print("\n OUTPUT FILES ")
for folder in ['checkpoints','plots','metrics']:
    path = f"{OUT}/{folder}"
    if os.path.exists(path):
        files = os.listdir(path)
        print(f"\n{folder}/ ({len(files)} files)")
        for fn in sorted(files):
            sz = os.path.getsize(f"{path}/{fn}")
            print(f"  {fn}  ({sz/1024:.0f} KB)")



MULTI-SEED RESULTS 
Model                                     AUC          ECE      Acc     Prec      Rec
ABMIL                             0.9155±0.0031  0.3553±0.0636  87.6667±0.4714  0.8852  0.8000
CLAM-SB                           0.9161±0.0035  0.2933±0.0807  89.0000±1.4142  0.9403  0.7750
TransMIL                          0.9161±0.0151  0.1891±0.1447  85.3333±4.6428  0.8635  0.7917
VAEABMIL                          0.9214±0.0038  0.3624±0.1226  88.3333±2.0548  0.8989  0.8000
eval_nogate (No Gating)           0.9204±0.0012  0.2312±0.0710  89.3333±0.4714  0.9469  0.7833
URAT-MIL (Proposed)               0.9186±0.0054  0.3178±0.0267  85.0000±3.5590  0.8630  0.7833
  Saved: /kaggle/working/plots/comparison_multiseed.png

 OUTPUT FILES 

checkpoints/ (18 files)
  abmil_seed123.pt  (842 KB)
  abmil_seed42.pt  (842 KB)
  abmil_seed456.pt  (842 KB)
  clam_seed123.pt  (777 KB)
  clam_seed42.pt  (777 KB)
  clam_seed456.pt  (777 KB)
  nogate_seed123.pt  (1749 KB)
  nogate_seed42.pt  (1749 K